In [14]:
from sqlalchemy import create_engine
import psycopg2
import pandas as pd

# Replace with your actual database credentials
username = 'postgres'
password = 'abcd1234'

host = 'localhost'  
port = '5432'              
database = 'MAPE forecast'

# Create the database connection
engine = create_engine(f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}')

# Query the database
query = "SELECT * FROM mapeforecast"
database_df = pd.read_sql(query, engine)

# Display the dataframe
database_df.head()

,Item Label,Month,Year,Quarter,Population,PriProdZone,Product,ProdCat,StorageCost,PetrolPrice,DieselPrice,ExchangeRate,GDP,Price
0,Agric eggs medium size,Jan,2017,Q1,197378751,South West,Livestock,Perishable,High,148.7,216.0,305.25,297903.4,512.99
1,Agric eggs(medium size price of one),Jan,2017,Q1,197378751,South West,Livestock,Perishable,High,148.7,216.0,305.25,297903.4,47.42
2,"Beans brown,sold loose",Jan,2017,Q1,197378751,North Central,Crops,Non-Perishable,Low,148.7,216.0,305.25,2943533.2,353.60
3,Beans:white black eye. sold loose,Jan,2017,Q1,197378751,North East,Crops,Non-Perishable,Low,148.7,216.0,305.25,2943533.2,305.53
4,Beef Bone in,Jan,2017,Q1,197378751,North West,Livestock,Perishable,High,148.7,216.0,305.25,297903.4,1001.24


In [4]:
import requests
import pandas as pd
import time

# Define the states with their coordinates and corresponding zones
states_info = [
    {"state": "Kogi", "lat": 7.8, "lon": 6.73, "zone": "North Central"},
    {"state": "Nasarawa", "lat": 8.5, "lon": 8.52, "zone": "North Central"},
    {"state": "Benue", "lat": 7.19, "lon": 8.13, "zone": "North Central"},
    {"state": "Borno", "lat": 11.83, "lon": 13.15, "zone": "North East"},
    {"state": "Gombe", "lat": 10.28, "lon": 11.17, "zone": "North East"},
    {"state": "Yobe", "lat": 12.0, "lon": 11.5, "zone": "North East"},
    {"state": "Kaduna", "lat": 10.52, "lon": 7.43, "zone": "North West"},
    {"state": "Kano", "lat": 12.0, "lon": 8.59, "zone": "North West"},
    {"state": "Sokoto", "lat": 13.06, "lon": 5.24, "zone": "North West"},
    {"state": "Enugu", "lat": 6.45, "lon": 7.51, "zone": "South East"},
    {"state": "Abia", "lat": 5.45, "lon": 7.52, "zone": "South East"},
    {"state": "Anambra", "lat": 6.22, "lon": 6.93, "zone": "South East"},
    {"state": "Delta", "lat": 5.70, "lon": 5.93, "zone": "South South"},
    {"state": "Akwa Ibom", "lat": 4.91, "lon": 7.85, "zone": "South South"},
    {"state": "Bayelsa", "lat": 4.77, "lon": 6.08, "zone": "South South"},
    {"state": "Lagos", "lat": 6.52, "lon": 3.38, "zone": "South West"},
    {"state": "Oyo", "lat": 7.38, "lon": 3.95, "zone": "South West"},
    {"state": "Osun", "lat": 7.77, "lon": 4.56, "zone": "South West"},
]

# Function to fetch daily precipitation data
def fetch_daily_precipitation(lat, lon, start_date="2017-01-01", end_date="2024-12-31"):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "precipitation_sum",
        "timezone": "Africa/Lagos"
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    if "daily" not in data or "time" not in data["daily"] or "precipitation_sum" not in data["daily"]:
        raise ValueError("Incomplete data received from API.")
    df = pd.DataFrame({
        "Date": pd.to_datetime(data["daily"]["time"]),
        "Rainfall (mm)": data["daily"]["precipitation_sum"]
    })
    return df

# Dictionary to hold DataFrames for each state
state_data = {}

for state in states_info:
    try:
        print(f"Fetching data for {state['state']}...")
        df = fetch_daily_precipitation(state["lat"], state["lon"])
        df["Month"] = df["Date"].dt.to_period("M")
        monthly_df = df.groupby("Month")["Rainfall (mm)"].sum().reset_index()
        monthly_df["State"] = state["state"]
        monthly_df["Zone"] = state["zone"]
        state_data[state["state"]] = monthly_df
        time.sleep(1)  # To respect API rate limits
    except Exception as e:
        print(f"Failed to fetch data for {state['state']}: {e}")

# Combine all state data into a single DataFrame
all_states_df = pd.concat(state_data.values(), ignore_index=True)

# Calculate mean monthly rainfall per zone
zone_monthly_mean = all_states_df.groupby(["Zone", "Month"])["Rainfall (mm)"].mean().reset_index()
zone_monthly_mean.rename(columns={"Rainfall (mm)": "Mean Rainfall (mm)"}, inplace=True)

# Display the first few rows
print(zone_monthly_mean.head())


Fetching data for Kogi...
Fetching data for Nasarawa...
Fetching data for Benue...
Fetching data for Borno...
Fetching data for Gombe...
Fetching data for Yobe...
Fetching data for Kaduna...
Fetching data for Kano...
Fetching data for Sokoto...
Fetching data for Enugu...
Fetching data for Abia...
Fetching data for Anambra...
Fetching data for Delta...
Fetching data for Akwa Ibom...
Fetching data for Bayelsa...
Fetching data for Lagos...
Fetching data for Oyo...
Fetching data for Osun...
            Zone    Month  Mean Rainfall (mm)
0  North Central  2017-01            0.566667
1  North Central  2017-02            0.000000
2  North Central  2017-03            3.866667
3  North Central  2017-04           36.900000
4  North Central  2017-05           63.200000


In [5]:
zone_monthly_mean.isna().sum()

Zone                  0
Month                 0
Mean Rainfall (mm)    0
dtype: int64

In [6]:
zone_monthly_mean['Zone'].value_counts()    

Zone
North Central    96
North East       96
North West       96
South East       96
South South      96
South West       96
Name: count, dtype: int64

In [7]:
zone_monthly_mean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 576 entries, 0 to 575
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype    
---  ------              --------------  -----    
 0   Zone                576 non-null    object   
 1   Month               576 non-null    period[M]
 2   Mean Rainfall (mm)  576 non-null    float64  
dtypes: float64(1), object(1), period[M](1)
memory usage: 13.6+ KB


In [8]:
zone_monthly_mean['Month'] = zone_monthly_mean['Month'].astype(str)
rain_df = zone_monthly_mean.copy()
rain_df['Year'] = rain_df['Month'].str.split('-').str[0].astype(int)
rain_df['Month'] = rain_df['Month'].str.split('-').str[1].astype(int)
rain_df.head()

,Zone,Month,Mean Rainfall (mm),Year
0,North Central,1,0.566667,2017
1,North Central,2,0.000000,2017
2,North Central,3,3.866667,2017
3,North Central,4,36.900000,2017
4,North Central,5,63.200000,2017


In [9]:
month_map = {
    1: 'Jan',
    2: 'Feb',
    3: 'Mar', 
    4: 'Apr',
    5: 'May',
    6: 'Jun',
    7: 'Jul',
    8: 'Aug',
    9: 'Sep',
    10: 'Oct',
    11: 'Nov',
    12: 'Dec'
}

rain_df['Month'] = rain_df['Month'].map(month_map)

In [18]:
rain_df.head()

,PriProdZone,Month,Mean Rainfall (mm),Year
0,North Central,Jan,0.566667,2017
1,North Central,Feb,0.000000,2017
2,North Central,Mar,3.866667,2017
3,North Central,Apr,36.900000,2017
4,North Central,May,63.200000,2017


In [17]:
rain_df.rename(columns={'Zone': 'PriProdZone'}, inplace=True)

In [15]:
database_df.head()

,Item Label,Month,Year,Quarter,Population,PriProdZone,Product,ProdCat,StorageCost,PetrolPrice,DieselPrice,ExchangeRate,GDP,Price
0,Agric eggs medium size,Jan,2017,Q1,197378751,South West,Livestock,Perishable,High,148.7,216.0,305.25,297903.4,512.99
1,Agric eggs(medium size price of one),Jan,2017,Q1,197378751,South West,Livestock,Perishable,High,148.7,216.0,305.25,297903.4,47.42
2,"Beans brown,sold loose",Jan,2017,Q1,197378751,North Central,Crops,Non-Perishable,Low,148.7,216.0,305.25,2943533.2,353.60
3,Beans:white black eye. sold loose,Jan,2017,Q1,197378751,North East,Crops,Non-Perishable,Low,148.7,216.0,305.25,2943533.2,305.53
4,Beef Bone in,Jan,2017,Q1,197378751,North West,Livestock,Perishable,High,148.7,216.0,305.25,297903.4,1001.24


In [19]:
final_df = database_df.merge(rain_df, on=['PriProdZone', 'Year', 'Month'], how='left')
final_df.head()

,Item Label,Month,Year,Quarter,Population,PriProdZone,Product,ProdCat,StorageCost,PetrolPrice,DieselPrice,ExchangeRate,GDP,Price,Mean Rainfall (mm)
0,Agric eggs medium size,Jan,2017,Q1,197378751,South West,Livestock,Perishable,High,148.7,216.0,305.25,297903.4,512.99,9.500000
1,Agric eggs(medium size price of one),Jan,2017,Q1,197378751,South West,Livestock,Perishable,High,148.7,216.0,305.25,297903.4,47.42,9.500000
2,"Beans brown,sold loose",Jan,2017,Q1,197378751,North Central,Crops,Non-Perishable,Low,148.7,216.0,305.25,2943533.2,353.60,0.566667
3,Beans:white black eye. sold loose,Jan,2017,Q1,197378751,North East,Crops,Non-Perishable,Low,148.7,216.0,305.25,2943533.2,305.53,0.000000
4,Beef Bone in,Jan,2017,Q1,197378751,North West,Livestock,Perishable,High,148.7,216.0,305.25,297903.4,1001.24,0.000000


In [20]:
final_df.isna().sum()

Item Label              0
Month                   0
Year                    0
Quarter                 0
Population              0
PriProdZone             0
Product                 0
ProdCat                 0
StorageCost             0
PetrolPrice             0
DieselPrice             0
ExchangeRate            0
GDP                     0
Price                   0
Mean Rainfall (mm)    752
dtype: int64

In [21]:
final_df[final_df['Mean Rainfall (mm)'].isna()]['PriProdZone'].value_counts()

PriProdZone
Imported    752
Name: count, dtype: int64

In [22]:
final_df['Mean Rainfall (mm)'].fillna(0, inplace=True)

C:\Users\Daye Erekosima\AppData\Local\Temp\ipykernel_6912\2009924166.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_df['Mean Rainfall (mm)'].fillna(0, inplace=True)


In [23]:
final_df['Mean Rainfall (mm)'].isna().sum()

0

In [24]:
final_df.to_csv('final_food_price_rainfall_data.csv', index=False)